In [8]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders.wikipedia import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from dotenv import load_dotenv

In [9]:
# <redacted>
#  https://openai.vocareum.com/v1
load_dotenv()


True

In [10]:
loader = WikipediaLoader(
    "Anthony_Hopkins",
    load_max_docs=1,
    doc_content_chars_max=40000
)
docs = loader.load()

len(docs)

1

In [19]:
len(docs[0].page_content)
#print(docs[0].page_content[:1000])

39849

In [20]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
all_splits = text_splitter.split_documents(docs)

In [22]:
print(f"Split Wikipedia page into {len(all_splits)} sub-documents.")
all_splits[0]

Split Wikipedia page into 188 sub-documents.


Document(metadata={'title': 'Anthony Hopkins', 'summary': "Sir Philip Anthony Hopkins (born 31 December 1937) is a Welsh actor. Considered one of Britain's most recognisable and prolific actors, he is known for his performances on the screen and stage. Hopkins has received numerous accolades, including two Academy Awards, four BAFTA Awards, two Primetime Emmy Awards, and a Laurence Olivier Award. He has also received the Cecil B. DeMille Award in 2005 and the BAFTA Fellowship for lifetime achievement in 2008. He was knighted by Queen Elizabeth II for his services to drama in 1993.\nAfter graduating from the Royal Welsh College of Music & Drama in 1957, Hopkins trained at RADA (the Royal Academy of Dramatic Art) in London. He was then spotted by Laurence Olivier, who invited him to join the Royal National Theatre in 1965. Productions at the National included King Lear (his favourite Shakespeare play), Coriolanus, Macbeth, and Antony and Cleopatra. In 1985, he received acclaim and a Laur

In [24]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embeddings)

In [27]:
document_ids = vector_store.add_documents(documents=all_splits)
document_ids[:5]

['e9404c23-cd13-44f8-828e-953a990fd6d6',
 'bfcb62dd-ef8a-4a35-ae67-344ff8a39249',
 '843881f3-66b4-4c81-8ee8-fdc56f21173d',
 '6dfe687b-35c8-4db0-993f-2dca4fa93334',
 'd64ad959-4c19-459e-b5c4-33b37626f2ae']

In [28]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [29]:
template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that answers questions."),
     ("human", "Use the following pieces of retrieved context to answer the question. "
              "If you don't know the answer, just say that you don't know. " 
              "Use three sentences maximum and keep the answer concise. "
              "\n# Question: \n-> {question} "
              "\n# Context: \n-> {context} "
              "\n# Answer: "),
])


In [30]:
template.invoke(
    {"context": "##CONTEXT##", "question": "##QUESTION##"}
).to_messages()

[SystemMessage(content='You are a helpful assistant that answers questions.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise. \n# Question: \n-> ##QUESTION## \n# Context: \n-> ##CONTEXT## \n# Answer: ", additional_kwargs={}, response_metadata={})]

In [31]:
def format_docs(docs):
    formatted = "\n\n-> ".join(doc.page_content for doc in docs)
    return formatted

In [32]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
)

In [33]:
question = "When The Silence of the Lambs was released?"
context = format_docs(retriever.invoke(question))
messages = template.invoke({'question' : question, 'context' : context}).to_messages()
answer = llm.invoke(messages)

In [ ]:
for m in messages:
    print(f"{m}\n")


content='You are a helpful assistant that answers questions.' additional_kwargs={} response_metadata={}

content="Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise. \n# Question: \n-> When The Silence of the Lambs was released? \n# Context: \n-> Hopkins won acclaim among critics and audiences as the cannibalistic serial killer Hannibal Lecter in The Silence of the Lambs, for which he won the Academy Award for Best Actor in 1991, with Jodie Foster as Clarice Starling, who also won for Best Actress. The film won Best Picture, Best Director\n\n-> Hopkins won acclaim among critics and audiences as the cannibalistic serial killer Hannibal Lecter in The Silence of the Lambs, for which he won the Academy Award for Best Actor in 1991, with Jodie Foster as Clarice Starling, who also won for Best Actress. The film won Best Picture, Best Director\n\n-> Hopkins wo

In [39]:
print(answer)

content='The Silence of the Lambs was released in 1991.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 285, 'total_tokens': 298, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e2d886d409', 'id': 'chatcmpl-Do9Nv7DYCCcypELOb3lrWroz6uYxn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ea29f-de78-7711-bb6c-0a5889b340ad-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 285, 'output_tokens': 13, 'total_tokens': 298, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [41]:
rag_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        question=RunnablePassthrough()
    )
    |  template
    |  llm
)

In [42]:
rag_chain.invoke("When The Silence of the Lambs was released?")

AIMessage(content='The Silence of the Lambs was released in 1991.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 285, 'total_tokens': 298, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e2d886d409', 'id': 'chatcmpl-Do9VfYtNOvTcCA9f1SvgpBgTy5zmR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ea2a7-321b-7a60-a0ce-5208ae068688-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 285, 'output_tokens': 13, 'total_tokens': 298, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})